# Synthesis + Performance Extraction Pipeline

This notebook demonstrates the step-by-step extraction of:
1. **Materials** from a scientific paper
2. **Synthesis procedures** for each material
3. **Performance data** from plots, linked to materials

Each step produces visible output so you can inspect intermediate results.

## Setup: Configuration

In [1]:
import sys
print(sys.executable)

/Users/valeriegentzke/Documents/Studieren/lemat_synth/lematerial-llm-synthesis/.venv/bin/python


In [2]:
# Force reload modules (run this after code changes)
import importlib
import llm_synthesis.config.plot_filter_config
importlib.reload(llm_synthesis.config.plot_filter_config)   

<module 'llm_synthesis.config.plot_filter_config' from '/Users/valeriegentzke/Documents/Studieren/lemat_synth/lematerial-llm-synthesis/src/llm_synthesis/config/plot_filter_config.py'>

In [3]:
# ==============================================================================
# USER CONFIGURATION - Edit these values
# ==============================================================================

# Path to your PDF or markdown file
INPUT_PATH = "/Users/valeriegentzke/Documents/Studieren/lemat_synth/lematerial-llm-synthesis/examples/data/pdf_papers/catalysis_corpus/Do2024Turning.pdf"  # or .md file with embedded images


# Output directory for results
OUTPUT_DIR = "results/"

# Models to use
GEMINI_MODEL = "gemini-3.0-flash"  # For synthesis extraction
CLAUDE_MODEL = "claude-sonnet-4-5-20250929" #"claude-sonnet-4-20250514"  # For plot data extraction
LINKER_MODEL = "gemini-3.0-flash"  # For series-to-material matching

# Set to True to skip figure/performance extraction (synthesis only)
SKIP_FIGURES = False

In [4]:
# Load environment and imports
import os
import sys
import json
import warnings
from pathlib import Path

# Add src directory to Python path (required for imports)
src_path = Path("../../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from dotenv import load_dotenv

# Load .env file
env_path = Path("../../.env")  # Adjust path if needed
load_dotenv(env_path, override=True)

# Silence noisy loggers
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

import logging
logging.getLogger("pydantic").setLevel(logging.ERROR)
logging.getLogger("LiteLLM").setLevel(logging.ERROR)
logging.getLogger("litellm").setLevel(logging.ERROR)

print("[OK] Environment loaded")
print(f"[OK] src path added: {src_path}")

[OK] Environment loaded
[OK] src path added: /Users/valeriegentzke/Documents/Studieren/lemat_synth/lematerial-llm-synthesis/src


## Step 0: Load Paper Text

Extract text from PDF using Mistral OCR. If you already have markdown, it loads directly.

In [5]:
from llm_synthesis.data_loader.paper_loader.fs_paper_loader import FSPaperLoader
from llm_synthesis.models.paper import Paper

# ==============================================================================
# SI FILE DETECTION HELPERS
# ==============================================================================

SI_PATTERNS = ["_SI", "-SI", "_si", "-si", "_Supporting", "_supporting",
               "_Supplementary", "_supplementary", "_supp", "_Supp"]


def find_si_file(main_paper_path: Path) -> Path | None:
    """Find the SI file matching a main paper."""
    parent_dir = main_paper_path.parent
    main_stem = main_paper_path.stem
    
    for pattern in SI_PATTERNS:
        for ext in [".pdf", ".md", ".txt"]:
            si_path = parent_dir / f"{main_stem}{pattern}{ext}"
            if si_path.exists():
                return si_path
    return None


def load_file_text(path: Path, pdf_extractor=None) -> str:
    """Load text from a PDF, MD, or TXT file."""
    suffix = path.suffix.lower()
    
    if suffix == ".pdf":
        if pdf_extractor is None:
            from llm_synthesis.transformers.pdf_extraction import MistralPDFExtractor
            pdf_extractor = MistralPDFExtractor(structured=False)
        with open(path, "rb") as f:
            return pdf_extractor.forward(f.read())
    elif suffix in [".md", ".txt"]:
        with open(path, "r", errors="replace") as f:
            return f.read()
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


# ==============================================================================
# LOAD MAIN PAPER
# ==============================================================================

input_path = Path(INPUT_PATH)
pdf_extractor = None

if input_path.suffix.lower() == ".pdf":
    print(f"Extracting text from PDF: {input_path.name}")
    from llm_synthesis.transformers.pdf_extraction import MistralPDFExtractor
    pdf_extractor = MistralPDFExtractor(structured=False)
    paper_text = load_file_text(input_path, pdf_extractor)
    print(f"   Main paper: {len(paper_text):,} characters")
    
elif input_path.suffix.lower() in [".md", ".txt"]:
    print(f"Loading markdown: {input_path.name}")
    paper_text = load_file_text(input_path)
    print(f"   Main paper: {len(paper_text):,} characters")
    
elif input_path.is_dir():
    print(f"Loading papers from directory: {input_path}")
    loader = FSPaperLoader(data_dir=str(input_path))
    papers = loader.load()
    paper = papers[0]
    print(f"   Loaded paper: {paper.name} ({len(paper.publication_text):,} characters)")
else:
    raise ValueError(f"Unsupported input type: {input_path}")

# ==============================================================================
# LOAD SI FILE (if exists)
# ==============================================================================

si_text = ""
if not input_path.is_dir():
    si_path = find_si_file(input_path)
    if si_path:
        print(f"   Found SI file: {si_path.name}")
        try:
            si_text = load_file_text(si_path, pdf_extractor)
            print(f"   SI text: {len(si_text):,} characters")
        except Exception as e:
            print(f"   [WARN] Failed to load SI file: {e}")

# ==============================================================================
# CREATE PAPER OBJECT
# ==============================================================================

if not input_path.is_dir():
    paper = Paper(
        name=input_path.stem,
        id=input_path.stem,
        publication_text=paper_text,
        si_text=si_text,
    )

print(f"\n[OK] Paper loaded successfully")
print(f"   Paper ID: {paper.id}")
print(f"   Paper Name: {paper.name}")
print(f"   Main text: {len(paper.publication_text):,} chars")
print(f"   SI text: {len(paper.si_text):,} chars")

Extracting text from PDF: Do2024Turning.pdf
   Main paper: 1,254,145 characters

[OK] Paper loaded successfully
   Paper ID: Do2024Turning
   Paper Name: Do2024Turning
   Main text: 1,254,145 chars
   SI text: 0 chars


In [6]:
# Preview paper text
print("=" * 60)
print("PAPER TEXT PREVIEW (first 2000 chars)")
print("=" * 60)
print(paper.publication_text[:2000])
print("...")

PAPER TEXT PREVIEW (first 2000 chars)
Chemical Engineering Journal 501 (2024) 157605

ELSEVIER

Contents lists available at ScienceDirect

Chemical Engineering Journal

journal homepage: www.elsevier.com/locate/cej

CEJ

# Turning properties of Ni/Al₂O₃ catalyst to improve catalytic ammonia decomposition for green hydrogen production: pH does matter!

Quoc Cuong Do a,1, Youngmin Kim a,1, Geo Jong Kim a,1, Younghwan Im a, Thien An Le a, Gye Hong Kim b, Kyoung Chul Ko c, Ho-Jeong Chae a,*

$^{a}$ Chemical &amp; Process Technology Division, Korea Research Institute of Chemical Technology, 141 Gajeong-ro, Yuseong-gu, Daejeon 34114, Republic of Korea
$^{b}$ Department of Earth Systems and Environmental Sciences, Chonnam National University, Gwangju 61186, Republic of Korea
$^{c}$ Department of Chemistry Education, Chonnam National University, Gwangju 61186, Republic of Korea

# ARTICLEINFO

Keywords:
Ammonia decomposition
Hydrogen production
Ru-free catalysts
Ni-based catalysts
Engineering 

## Step 1: Extract Materials

Identify all materials that were synthesized in this paper.

In [7]:
from llm_synthesis.transformers.material_extraction.dspy_extraction import (
    DspyTextExtractor,
    make_dspy_text_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name
from llm_synthesis.utils import clean_text

# Create material extractor
material_sig = make_dspy_text_extractor_signature(
    instructions=(
        "Extract ALL distinct material compositions that were synthesized and tested in this paper. "
        "IMPORTANT: If the paper studies multiple variants of a material (e.g., different loadings, "
        "dopant concentrations, or preparation conditions), list EACH variant as a separate material. "
        "For example, if a paper studies 1%Ru/CaO, 3%Ru/CaO, and 5%Ru/CaO, list all three - "
        "do NOT merge them into a single 'Ru/CaO'. "
        "Focus on materials that were actually synthesized, not just mentioned or referenced."
    ),
    output_description=(
        "ALL distinct synthesized material compositions as a comma-separated list using chemical formulas. "
        "Include loading percentages and promoters when specified "
        "(e.g., '1%Ru-10%K/CaO, 3%Ru-10%K/CaO, 5%Ru-10%K/CaO, 3%Ru-5%K/CaO'). "
        "Never merge variants into a single generic name."
    ),
)

material_lm = get_llm_from_name(
    "gemini-3.0-pro",
    model_kwargs={"temperature": 0.0, "max_tokens": 8000},
)

material_extractor = DspyTextExtractor(signature=material_sig, lm=material_lm)

print("Extracting materials...")





Extracting materials...


In [8]:
# Run material extraction
materials_text = material_extractor.forward(input=clean_text(paper.publication_text))

# Parse into list
materials = [
    m.strip()
    for m in materials_text.replace("\n", ",").split(",")
    if m.strip()
]

print("=" * 60)
print(f"MATERIALS FOUND ({len(materials)} total)")
print("=" * 60)
for i, mat in enumerate(materials, 1):
    print(f"  {i}. {mat}")

2026/02/10 11:40:38 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


MATERIALS FOUND (12 total)
  1. 10Ni/Al₂O₃ (pH 6.0)
  2. 10Ni/Al₂O₃ (pH 7.0)
  3. 10Ni/Al₂O₃ (pH 8.0)
  4. 10Ni/Al₂O₃ (pH 9.0)
  5. 10Ni/Al₂O₃ (pH 10.0)
  6. 10Ni/Al₂O₃ (pH 11.0)
  7. 10Ni/Al₂O₃ (pH 12.5)
  8. 20Ni/Al₂O₃
  9. 40Ni/Al₂O₃
  10. 60Ni/Al₂O₃
  11. 80Ni/Al₂O₃
  12. NiAl₂O₄


## Step 2: Extract Synthesis Procedures

For each material, extract the detailed synthesis procedure.

In [11]:
from llm_synthesis.transformers.synthesis_extraction.dspy_synthesis_extraction import (
    DspySynthesisExtractor,
    make_dspy_synthesis_extractor_signature,
)
from llm_synthesis.metrics.judge.general_synthesis_judge import (
    DspyGeneralSynthesisJudge,
    make_general_synthesis_judge_signature,
)

# System prompt for synthesis extraction
SYNTHESIS_SYSTEM_PROMPT = """You are a helpful assistant that extracts structured synthesis procedures from scientific papers.

IMPORTANT: For the synthesis_method field, you MUST choose from these exact values:
'PVD', 'CVD', 'arc discharge', 'ball milling', 'spray pyrolysis', 'electrospinning',
'sol-gel', 'hydrothermal', 'solvothermal', 'precipitation', 'coprecipitation', 'combustion',
'microwave-assisted', 'sonochemical', 'template-directed', 'solid-state', 'flux growth',
'float zone & Bridgman', 'arc melting & induction melting', 'spark plasma sintering',
'electrochemical deposition', 'chemical bath deposition', 'liquid-phase epitaxy', 'self-assembly',
'atomic layer deposition', 'molecular beam epitaxy', 'pulsed laser deposition', 'ion implantation',
'lithographic patterning', 'wet impregnation', 'incipient wetness impregnation', 'mechanical mixing',
'solution-based', 'mechanochemical', 'other'

For the target_compound_type field, you MUST choose from these exact values:
'metals & alloys', 'ceramics & glasses', 'polymers & soft matter', 'composites',
'semiconductors & electronic', 'nanomaterials', 'two-dimensional materials',
'framework & porous materials', 'biomaterials & biological', 'liquid materials',
'hybrid & organic-inorganic', 'functional materials & catalysts', 'energy & sustainability',
'smart & responsive materials', 'emerging & quantum materials', 'other'

If the exact method is not in the list, use the closest match or 'other'."""

# Create synthesis extractor
synthesis_sig = make_dspy_synthesis_extractor_signature(
    instructions=(
        "Extract the complete structured synthesis procedure for the specified material. "
        "Include all steps, conditions (temperature, time, atmosphere), equipment, and precursors. "
        "Be thorough and preserve all quantitative details."
    ),
)

synthesis_lm = get_llm_from_name(
    GEMINI_MODEL,
    model_kwargs={"temperature": 0.0, "max_tokens": 32000, "max_retries": 3},
    system_prompt=SYNTHESIS_SYSTEM_PROMPT,
)
synthesis_extractor = DspySynthesisExtractor(signature=synthesis_sig, lm=synthesis_lm)

# Create judge
judge_lm = get_llm_from_name(
    GEMINI_MODEL,
    model_kwargs={"temperature": 0.1, "max_tokens": 4096},
)
judge_sig = make_general_synthesis_judge_signature()
judge = DspyGeneralSynthesisJudge(signature=judge_sig, lm=judge_lm)

print("[OK] Synthesis extractor and judge initialized")

[OK] Synthesis extractor and judge initialized


In [12]:
# Extract synthesis for each material
from llm_synthesis.models.paper import SynthesisEntry

all_syntheses = []
text_for_llm = clean_text(paper.publication_text)

for i, material in enumerate(materials, 1):
    print(f"\n{'=' * 60}")
    print(f"EXTRACTING SYNTHESIS {i}/{len(materials)}: {material}")
    print("=" * 60)
    
    try:
        # Extract synthesis
        synthesis = synthesis_extractor.forward(input=(text_for_llm, material))
        
        # Evaluate
        try:
            evaluation = judge.forward(
                (text_for_llm, json.dumps(synthesis.model_dump()), material)
            )
            print(f"   [OK] Evaluation score: {evaluation.scores.overall_score}/5.0")
        except Exception as e:
            print(f"   [WARN] Judge failed: {e}")
            evaluation = None
        
        all_syntheses.append(SynthesisEntry(
            material=material,
            synthesis=synthesis,
            evaluation=evaluation,
        ))
        
        # Show synthesis summary
        print(f"\n   Target: {synthesis.target_compound}")
        print(f"   Type: {synthesis.target_compound_type}")
        print(f"   Method: {synthesis.synthesis_method}")
        print(f"   Starting materials: {len(synthesis.starting_materials)}")
        print(f"   Steps: {len(synthesis.steps)}")
        
    except Exception as e:
        print(f"   [ERROR] Extraction failed: {e}")
        all_syntheses.append(SynthesisEntry(
            material=material,
            synthesis=None,
            evaluation=None,
        ))

print(f"\n\n[OK] Extracted synthesis for {len(all_syntheses)} materials")

2026/02/10 11:43:50 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.



EXTRACTING SYNTHESIS 1/12: 10Ni/Al₂O₃ (pH 6.0)


2026/02/10 11:46:40 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.

LM Response: {
        "name": "magnetic stirrer",
        "settings": "vigorous stirring"
      } 

Expected to find output fields in the LM response: [structured_synthesis] 

Actual output fields parsed from the LM response: [] 


2026/02/10 11:46:40 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/10 11:46:40 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


   [OK] Evaluation score: 1.0/5.0

   Target: 10Ni/Al₂O₃ (pH 6.0)
   Type: other
   Method: other
   Starting materials: 0
   Steps: 0

EXTRACTING SYNTHESIS 2/12: 10Ni/Al₂O₃ (pH 7.0)


2026/02/10 11:47:26 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/10 11:47:48 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


...

   [OK] Evaluation score: 5.0/5.0

   Target: 60Ni/Al2O3
   Type: functional materials & catalysts
   Method: coprecipitation
   Starting materials: 5
   Steps: 10

EXTRACTING SYNTHESIS 11/12: 80Ni/Al₂O₃


2026/02/10 11:59:19 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/10 11:59:35 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


   [OK] Evaluation score: 4.9/5.0

   Target: 80Ni/Al2O3
   Type: functional materials & catalysts
   Method: precipitation
   Starting materials: 5
   Steps: 10

EXTRACTING SYNTHESIS 12/12: NiAl₂O₄


2026/02/10 12:00:08 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/10 12:00:34 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4096. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.


   [WARN] Judge failed: Both structured output format and JSON mode failed. Please choose a model that supports `response_format` argument. Original error: 15 validation errors for GeneralSynthesisEvaluation
scores.structural_completeness_reasoning
  Field required [type=missing, input_value={'structural_completeness_score': 5.0}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
scores.material_extraction_score
  Field required [type=missing, input_value={'structural_completeness_score': 5.0}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
scores.material_extraction_reasoning
  Field required [type=missing, input_value={'structural_completeness_score': 5.0}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
scores.process_steps_score
  Field required [type=missing, input_value={'structural_completeness_score': 5.0}, input_type=dict]
    For further inf

In [ ]:
# Detailed view of first synthesis
if all_syntheses and all_syntheses[0].synthesis:
    s = all_syntheses[0].synthesis
    print("=" * 60)
    print(f"DETAILED SYNTHESIS: {all_syntheses[0].material}")
    print("=" * 60)
    
    print(f"\nStarting Materials:")
    for mat in s.starting_materials:
        amt = f"{mat.amount} {mat.unit}" if mat.amount else "N/A"
        print(f"  - {mat.name}: {amt}")
    
    print(f"\nSynthesis Steps:")
    for step in s.steps:
        print(f"  Step {step.step_number}: {step.action}")
        if step.description:
            print(f"     {step.description[:100]}..." if len(step.description) > 100 else f"     {step.description}")
        if step.conditions:
            c = step.conditions
            cond_parts = []
            if c.temperature: cond_parts.append(f"{c.temperature} {c.temp_unit or 'C'}")
            if c.duration: cond_parts.append(f"{c.duration} {c.time_unit or 'h'}")
            if c.atmosphere: cond_parts.append(c.atmosphere)
            if cond_parts:
                print(f"     Conditions: {', '.join(cond_parts)}")

DETAILED SYNTHESIS: Co/MCM-41

Starting Materials:
  - MCM-41: N/A
  - Co(NO3)2·6H2O: N/A
  - water: N/A

Synthesis Steps:
  Step 1: impregnate
     MCM-41 was impregnated by incipient wetness impregnation with an aqueous solution of Co(NO3)2·6H2O. ...
  Step 2: dry
     The catalyst was dried in stagnant air at room temperature.
     Conditions: 12.0 h, stagnant air
  Step 3: dry
     The catalyst was dried in an oven.
     Conditions: 120.0 C, 12.0 h
  Step 4: calcine
     The dried catalyst was calcinated.
     Conditions: 600.0 C, 3.0 h
  Step 5: other
     The catalyst was stored in a sealed container.
  Step 6: reduce
     Before being used for ammonia decomposition, the catalyst was reduced in a stream of H2 (50 mL/min).
     Conditions: 600.0 C, 3.0 h, H2


: 

: 

: 

## Step 3: Extract Figures

Find and segment all figures in the paper into subfigures.

Two segmentation backends are available:
- **Florence-2**: Uses Florence-2 with LoRA adapter (single model, faster)
- **DINO**: Uses Grounding DINO + ResNet classifier (more granular classes)

All figures will be passed to Claude VLM for plot data extraction - Claude will determine which figures contain extractable numerical data.

In [ ]:
if SKIP_FIGURES:
    print("[SKIP] Skipping figure extraction (SKIP_FIGURES=True)")
    figures = []
else:
    from llm_synthesis.transformers.figure_extraction import FigureExtractorMarkdown
    
    # ==============================================================================
    # CHOOSE SEGMENTATION BACKEND
    # ==============================================================================
    
    # Option 1: Florence-2 with LoRA (recommended - single model, faster)
    extractor = FigureExtractorMarkdown(
        segmenter="florence",
        florence_repo_id="amayuelas/plot-visualization-florence-2-lora-32",
    )
    print("Extracting and segmenting figures using Florence-2...")
    
    # Option 2: DINO + ResNet (more granular figure classes)
    # extractor = FigureExtractorMarkdown(segmenter="dino")
    # print("Extracting and segmenting figures using DINO...")
    
    # ==============================================================================
    
    figures = extractor.forward(paper.publication_text)
    
    print(f"\n{'=' * 60}")
    print(f"FIGURES FOUND ({len(figures)} subfigures)")
    print("=" * 60)
    print("Note: All figures will be sent to Claude VLM for plot extraction")
    for i, fig in enumerate(figures):
        print(f"  {i+1}. {fig.figure_reference or f'Figure {i}'}: {fig.figure_class}")

Loading Florence-2 base model: microsoft/Florence-2-base-ft
Loading LoRA adapters from: amayuelas/plot-visualization-florence-2-lora-32
Merging LoRA adapters with base model...
Florence-2 model loaded on mps
Extracting and segmenting figures using Florence-2...
Found 43 figures in the paper.
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 8 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segmented 1 subfigures (Florence).
Segment

: 

: 

: 

In [ ]:
# Debug: Check context for each figure
print("=" * 60)
print("DEBUG: FIGURE CONTEXT INSPECTION")
print("=" * 60)

for i, fig in enumerate(figures):
    print(f"\n--- Figure {i+1}: {fig.figure_reference or 'Unknown'} ---")
    print(f"Context BEFORE ({len(fig.context_before)} chars):")
    print(fig.context_before[-300:] if len(fig.context_before) > 300 else fig.context_before)
    print(f"\nContext AFTER ({len(fig.context_after)} chars):")
    print(fig.context_after[:300] if len(fig.context_after) > 300 else fig.context_after)
    print("-" * 40)


DEBUG: FIGURE CONTEXT INSPECTION

--- Figure 1: Unknown Figure ---
Context BEFORE (500 chars):
 of Science and Technology, Trondheim 7491, Norway

## HIGHLIGHTS

- MCM-41 acts as nano-reactor to confine metallic nanoparticles.
- Co—Mo bimetallic catalyst shows a strong synergistic effect for $\mathrm{NH}_3$ decomposition.
- The optimum Co/Mo atomic ratio is about 7/3.

## GRAPHICAL ABSTRACT



Context AFTER (500 chars):
![fig](placeholder_image)

## ARTICLE INFO

Article history:
Available online 5 June 2012

Keywords:
Ammonia decomposition
Hydrogen production
Co—Mo bimetallic catalysts
Synergistic effect
MCM-41

## ABSTRACT

MCM-41 supported Co, Mo and Co—Mo bimetallic catalysts with different Co/Mo atomic ratios 
----------------------------------------

--- Figure 2: Fig. 2 ---
Context BEFORE (500 chars):
| 0.68 | 3.4  |
|  Co6Mo1/MCM-41 | 761.4 | 0.74 | 3.1  |
|  Co8Mo2/MCM-41 | 765.3 | 0.83 | 3.5  |
|  Co7Mo3/MCM-41 | 668.2 | 0.69 | 3.4  |
|  Co6Mo4/MCM-41 | 649.8 | 0.67 | 3.4  |


: 

: 

: 

## Step 4: Extract Plot Data

Use Claude VLM to extract numerical data from quantitative plots.

In [47]:
if SKIP_FIGURES or not figures:
    print("[SKIP] Skipping plot data extraction")
    plots = []
    plot_figures = []
else:
    from llm_synthesis.transformers.plot_extraction.claude_extraction.plot_data_extraction import (
        ClaudeLinePlotDataExtractor,
    )
    from llm_synthesis.models.figure import FigureInfoWithPaper
    from llm_synthesis.utils.figure_utils import clean_text_from_images
    
    print(f"Extracting data from {len(figures)} figures using Claude VLM...")
    print("(Claude will determine which figures contain extractable plot data)")
    
    plot_extractor = ClaudeLinePlotDataExtractor(model_name=CLAUDE_MODEL)
    
    plots = []
    plot_figures = []
    
    for i, fig in enumerate(figures):
        print(f"\n  Processing figure {i+1}/{len(figures)}: {fig.figure_reference or f'Figure {i}'} ({fig.figure_class})")
        
        fig_with_paper = FigureInfoWithPaper(
            base64_data=fig.base64_data,
            alt_text=fig.alt_text,
            position=fig.position,
            context_before=fig.context_before,
            context_after=fig.context_after,
            figure_reference=fig.figure_reference,
            figure_class=fig.figure_class,
            quantitative=fig.quantitative,
            paper_text=clean_text_from_images(paper.publication_text),
            si_text=paper.si_text,
        )
        
        try:
            plot_data = plot_extractor.forward(fig_with_paper)
            if plot_data and plot_data.name_to_coordinates:
                plots.append(plot_data)
                plot_figures.append(fig)
                print(f"    [OK] Extracted {len(plot_data.name_to_coordinates)} series")
                print(f"       Series: {list(plot_data.name_to_coordinates.keys())}")
            else:
                print(f"    [--] No extractable data (not a quantitative plot)")
        except Exception as e:
            print(f"    [ERROR] Failed: {e}")
    
    print(f"\n[OK] Extracted data from {len(plots)} plots out of {len(figures)} figures")

Extracting data from 56 figures using Claude VLM...
(Claude will determine which figures contain extractable plot data)

  Processing figure 1/56: Fig. 1 (unknown)
    [OK] Extracted 4 series
       Series: ['Series_1 (black circles)', 'Series_2 (red squares)', 'Series_3 (blue triangles)', 'Series_4 (green inverted triangles)']

  Processing figure 2/56: Fig. 2 (unknown)
    [--] No extractable data (not a quantitative plot)

  Processing figure 3/56: Fig. 3 (unknown)
    [OK] Extracted 8 series
       Series: ['3%Sm/Ni/Al₂O₃', '3%Nd/Ni/Al₂O₃', '3%Ce/Ni/Al₂O₃', '3%La/Ni/Al₂O₃', '50%Ni/Al₂O₃', 'NiO', 'NiAl₂O₄', 'Al₂O₃']

  Processing figure 4/56: Fig. 3 (unknown)
    [OK] Extracted 4 series
       Series: ['5%Sm/Ni/Al₂O₃', '5%Nd/Ni/Al₂O₃', '5%Ce/Ni/Al₂O₃', '5%La/Ni/Al₂O₃']

  Processing figure 5/56: Fig. 3 (unknown)
    [OK] Extracted 13 series
       Series: ['Al₂O₃ JCPDF 01-080-0956', 'Al₂O₃', 'NiAl₂O₃ JCPDF 10-0339', 'NiO JCPDF 01-071-1179', 'Ni/Al₂O₃', '10%La/Ni/Al₂O₃', 'La₂O₃', 'Ce

KeyboardInterrupt: 

In [ ]:
# Show plot details
if plots:
    print("=" * 60)
    print("EXTRACTED PLOT DATA SUMMARY")
    print("=" * 60)
    for i, (plot, fig) in enumerate(zip(plots, plot_figures)):
        print(f"\nPlot {i}: {fig.figure_reference or 'N/A'}")
        print(f"  Title: {plot.title or 'N/A'}")
        print(f"  X-axis: {plot.x_axis_label} [{plot.x_axis_unit}]")
        print(f"  Y-axis: {plot.y_left_axis_label} [{plot.y_left_axis_unit}]")
        print(f"  Series ({len(plot.name_to_coordinates)}):")
        for series_name, coords in plot.name_to_coordinates.items():
            print(f"    - {series_name}: {len(coords)} points")

EXTRACTED PLOT DATA SUMMARY

Plot 0: Unknown Figure
  Title: Temperature: 550°C
  X-axis: n_Co/(n_Co+n_Mo) []
  Y-axis: NH₃ conversion [%]
  Series (1):
    - Bar_data: 8 points

Plot 1: Fig. 2
  Title: N/A
  X-axis: None [None]
  Y-axis: None [None]
  Series (6):
    - Co/MCM-41: 17 points
    - Co9Mo1/MCM-41: 17 points
    - Co8Mo2/MCM-41: 17 points
    - Co7Mo3/MCM-41: 17 points
    - Co6Mo4/MCM-41: 17 points
    - Co5Mo5/MCM-41: 17 points

Plot 2: Fig. 3
  Title: N/A
  X-axis: None [None]
  Y-axis: None [None]
  Series (5):
    - (a) Co/MCM-41: 21 points
    - (b) Co9Mo1/MCM-41: 21 points
    - (c) Co8Mo2/MCM-41: 21 points
    - (d) Co7Mo3/MCM-41: 21 points
    - (e) Co6Mo4/MCM-41: 21 points

Plot 3: Fig. 5
  Title: N/A
  X-axis: 2-Theta [°]
  Y-axis:  []
  Series (1):
    - Co₇Mo₃/MCM-41: 56 points

Plot 4: Fig. 5
  Title: N/A
  X-axis: Binding energy [eV]
  Y-axis: Intensity [a.u.]
  Series (2):
    - Mo3d_5/2: 13 points
    - Mo3d_3/2: 11 points

Plot 5: Fig. 6
  Title: N/A
  X-

: 

: 

: 

## Step 5: Configure Plot Filtering (Optional)

Configure which plots should be included in performance linking based on axis characteristics.

In [ ]:
from llm_synthesis.config.plot_filter_config import PlotFilterConfig
from llm_synthesis.transformers.performance_linking.plot_filter import PlotFilter

# ==============================================================================
# CONFIGURE PLOT FILTERING
# ==============================================================================

# Option 1: Default catalysis config (temperature x-axis, conversion/yield y-axis)
# Filters for plots showing performance metrics (conversion, yield, selectivity)
# against temperature - the most common format in catalysis papers.
filter_config = PlotFilterConfig.for_catalysis()

# Option 2: Electrochemistry config (potential x-axis, current/capacitance y-axis)
# filter_config = PlotFilterConfig.for_electrochemistry()

# Option 3: No filtering (link all plots)
# filter_config = PlotFilterConfig.no_filter()

# Option 4: Custom config
# filter_config = PlotFilterConfig(
#     x_axis_labels=["temperature", "temp", "time"],  # Substring matching
#     x_axis_units=["k", "c", "h", "min"],
#     y_axis_keywords=["conversion", "yield", "selectivity"],
#     y_axis_units=["%"],
# )

plot_filter = PlotFilter(filter_config)

print("Plot Filter Configuration:")
print(f"   X-axis labels (substring match): {filter_config.x_axis_labels}")
print(f"   X-axis units: {filter_config.x_axis_units}")
print(f"   Y-axis keywords: {filter_config.y_axis_keywords}")
print(f"   Y-axis units: {filter_config.y_axis_units}")

Plot Filter Configuration:
   X-axis labels (substring match): ['temperature', 'temp']
   X-axis units: ['°c', '°k', '°f', 'ºc', 'ºk', 'k', 'c', 'f', 'kelvin', 'celsius']
   Y-axis keywords: ['conversion', 'yield', 'activity']
   Y-axis units: ['%', 'percent']


: 

: 

: 

In [ ]:
# Apply filtering
if plots:
    relevant_plots, skip_counts = plot_filter.filter_plots(plots, log_skipped=True)
    
    print(f"\n{'=' * 60}")
    print("PLOT FILTERING RESULTS")
    print("=" * 60)
    print(f"  Total plots: {len(plots)}")
    print(f"  Relevant plots: {len(relevant_plots)}")
    print(f"  Skipped (not relevant x-axis): {skip_counts.get('not_relevant_x', 0)}")
    print(f"  Skipped (not relevant y-axis): {skip_counts.get('not_relevant_y', 0)}")
    print(f"  Skipped (no series): {skip_counts.get('no_series', 0)}")
    
    print(f"\n  Relevant plots for linking:")
    for idx, plot in relevant_plots:
        print(f"    Plot {idx}: {plot.title or 'N/A'} ({len(plot.name_to_coordinates)} series)")
else:
    print("[SKIP] No plots to filter")
    relevant_plots = []


PLOT FILTERING RESULTS
  Total plots: 8
  Relevant plots: 2
  Skipped (not relevant x-axis): 6
  Skipped (not relevant y-axis): 0
  Skipped (no series): 0

  Relevant plots for linking:
    Plot 5: N/A (8 series)
    Plot 6: N/A (2 series)


: 

: 

: 

## Step 6: Link Series to Materials

Use LLM to match plot series names to extracted materials.

In [ ]:
if SKIP_FIGURES or not relevant_plots:
    print("[SKIP] Skipping performance linking")
    plot_mappings = []
    linking_stats = None
else:
    import dspy
    from llm_synthesis.transformers.performance_linking.series_material_linker import (
        SeriesMaterialLinker,
    )
    from llm_synthesis.transformers.performance_linking.base import LinkingInput
    from llm_synthesis.models.performance import PlotMaterialMapping
    
    print(f"Linking plot series to {len(materials)} materials...")
    
    # Initialize linker
    gemini_key = os.getenv("GEMINI_API_KEY", "")
    linker_lm = dspy.LM(
        f"gemini/{LINKER_MODEL}",
        temperature=0.0,
        max_tokens=8000,
        api_key=gemini_key,
    )
    series_linker = SeriesMaterialLinker(lm=linker_lm)
    
    plot_mappings = []
    
    for idx, plot in relevant_plots:
        fig = plot_figures[idx]
        series_names = list(plot.name_to_coordinates.keys())
        
        print(f"\n  Linking plot {idx}: '{plot.title or 'N/A'}' ({len(series_names)} series)")
        print(f"    Series: {series_names}")
        
        context = f"{fig.context_before} {fig.context_after}"
        plot_meta = {
            "title": plot.title,
            "x_axis_label": plot.x_axis_label,
            "x_axis_unit": plot.x_axis_unit,
            "y_left_axis_label": plot.y_left_axis_label,
            "y_left_axis_unit": plot.y_left_axis_unit,
        }
        
        # Call linker
        linking_input = LinkingInput(
            materials=materials,
            series_names=series_names,
            context=context,
            plot_metadata=plot_meta,
        )
        validated_mappings = series_linker.forward(linking_input)
        
        # Determine unmatched
        matched_series = {m.series_name for m in validated_mappings}
        unmatched = [s for s in series_names if s not in matched_series]
        
        plot_mappings.append(PlotMaterialMapping(
            plot_index=idx,
            figure_reference=fig.figure_reference,
            mappings=validated_mappings,
            unmatched_series=unmatched,
        ))
        
        print(f"    [OK] Matched: {len(validated_mappings)}")
        for m in validated_mappings:
            print(f"       '{m.series_name}' -> '{m.material_name}' ({m.confidence})")
        if unmatched:
            print(f"    [WARN] Unmatched: {unmatched}")
    
    print(f"\n[OK] Linking complete")

Linking plot series to 9 materials...

  Linking plot 5: 'N/A' (8 series)
    Series: ['Co/MCM-41', 'Co₉Mo₁/MCM-41', 'Co₈Mo₂/MCM-41', 'Co₇Mo₃/MCM-41', 'Co₆Mo₄/MCM-41', 'Co₃Mo₅/MCM-41', 'Co₃Mo₇/MCM-41', 'Mo/MCM-41']
    [OK] Matched: 7
       'Co/MCM-41' -> '5.0 wt.% Co/MCM-41' (high)
       'Co₉Mo₁/MCM-41' -> '5.0 wt.% Co9Mo1/MCM-41' (high)
       'Co₈Mo₂/MCM-41' -> '5.0 wt.% Co8Mo2/MCM-41' (high)
       'Co₇Mo₃/MCM-41' -> '5.0 wt.% Co7Mo3/MCM-41' (high)
       'Co₆Mo₄/MCM-41' -> '5.0 wt.% Co6Mo4/MCM-41' (high)
       'Co₃Mo₇/MCM-41' -> '5.0 wt.% Co3Mo7/MCM-41' (high)
       'Mo/MCM-41' -> '5.0 wt.% Mo/MCM-41' (high)
    [WARN] Unmatched: ['Co₃Mo₅/MCM-41']

  Linking plot 6: 'N/A' (2 series)
    Series: ['Co₇Mo₃/MCM-41', 'Co₇Mo₃/SiO₂']
    [OK] Matched: 2
       'Co₇Mo₃/MCM-41' -> '5.0 wt.% Co7Mo3/MCM-41' (high)
       'Co₇Mo₃/SiO₂' -> '5.0 wt.% Co7Mo3/SiO2' (high)

[OK] Linking complete


: 

: 

: 

## Step 7: Aggregate Performance Data

Combine all performance data per material.

In [ ]:
from llm_synthesis.utils.performance_utils import aggregate_all_materials_performance

if plot_mappings and plots:
    performance_data = aggregate_all_materials_performance(materials, plot_mappings, plots)
    
    print("=" * 60)
    print("PER-MATERIAL PERFORMANCE SUMMARY")
    print("=" * 60)
    
    for mat in materials:
        if mat in performance_data:
            perf = performance_data[mat]
            print(f"\n{mat}: {len(perf.plot_data)} performance entries")
            for entry in perf.plot_data:
                print(
                    f"  - {entry.plot_title or 'N/A'} / series '{entry.series_name}' "
                    f"({entry.y_axis_label} [{entry.y_axis_unit}]), "
                    f"{len(entry.coordinates)} points, confidence: {entry.confidence}"
                )
        else:
            print(f"\n{mat}: (no performance data linked)")
else:
    performance_data = {}
    print("[SKIP] No performance data to aggregate")

PER-MATERIAL PERFORMANCE SUMMARY

Co/MCM-41: 1 performance entries
  - N/A / series 'Co/MCM-41' (NH₃ conversion [%]), 5 points, confidence: high

Mo/MCM-41: 1 performance entries
  - N/A / series 'Mo/MCM-41' (NH₃ conversion [%]), 5 points, confidence: high

Co9Mo1/MCM-41: 1 performance entries
  - N/A / series 'Co₉Mo₁/MCM-41' (NH₃ conversion [%]), 5 points, confidence: high

Co8Mo2/MCM-41: (no performance data linked)

Co7Mo3/MCM-41: 2 performance entries
  - N/A / series 'Co₇Mo₃/MCM-41' (NH₃ conversion [%]), 5 points, confidence: high
  - N/A / series 'Co₇Mo₃/MCM-41' (NH₃ conversion [%]), 5 points, confidence: high

Co6Mo4/MCM-41: 1 performance entries
  - N/A / series 'Co₉Mo₄/MCM-41' (NH₃ conversion [%]), 5 points, confidence: medium

Co5Mo5/MCM-41: 1 performance entries
  - N/A / series 'Co₃Mo₅/MCM-41' (NH₃ conversion [%]), 5 points, confidence: medium

Co3Mo7/MCM-41: 1 performance entries
  - N/A / series 'Co₃Mo₇/MCM-41' (NH₃ conversion [%]), 5 points, confidence: high

Co7Mo3/SiO2

: 

: 

: 

## Step 7.5: Evaluate Linking Quality (LLM Judge)

Use an LLM judge to evaluate the quality of the synthesis-to-performance linking.
Scores 4 criteria (1-5) and flags 9 specific failure modes.

In [ ]:
from llm_synthesis.metrics.judge.linking_judge import (
    DspyLinkingJudge,
    make_linking_judge_signature,
)

linking_evaluation = None

if plot_mappings and performance_data:
    print("Evaluating linking quality with LLM judge...")

    # Initialize linking judge (same LM as synthesis judge)
    linking_judge_lm = get_llm_from_name(
        GEMINI_MODEL,
        model_kwargs={"temperature": 0.1, "max_tokens": 4096},
    )
    linking_judge_sig = make_linking_judge_signature()
    linking_judge = DspyLinkingJudge(signature=linking_judge_sig, lm=linking_judge_lm)

    # Prepare inputs
    synthesis_json = json.dumps(
        [
            {
                "material": e.material,
                "synthesis": e.synthesis.model_dump() if e.synthesis else None,
            }
            for e in all_syntheses
        ],
        indent=2,
    )
    plot_data_json = json.dumps(
        [p.model_dump() for p in plots],
        indent=2,
    )
    linking_output_json = json.dumps(
        {
            "mappings": [m.model_dump() for m in plot_mappings],
            "performance_per_material": {
                k: v.model_dump() for k, v in performance_data.items()
            },
        },
        indent=2,
    )

    try:
        linking_evaluation = linking_judge.forward(
            (clean_text(paper.publication_text), synthesis_json, plot_data_json, linking_output_json)
        )

        print(f"\n{'=' * 60}")
        print("LINKING EVALUATION RESULTS")
        print("=" * 60)
        scores = linking_evaluation.scores
        print(f"  Material Identity Match:       {scores.material_identity_score}/5.0")
        print(f"  Performance Data Correctness:  {scores.performance_data_correctness_score}/5.0")
        print(f"  Completeness:                  {scores.completeness_score}/5.0")
        print(f"  Format & Structure:            {scores.format_structure_score}/5.0")
        print(f"  Overall:                       {scores.overall_score}/5.0")
        print(f"  Confidence:                    {linking_evaluation.confidence_level}")

        active_flags = linking_evaluation.failure_flags.active_flags()
        if active_flags:
            print(f"\n  Failure flags: {active_flags}")
        else:
            print(f"\n  Failure flags: None")

        if linking_evaluation.missing_links:
            print(f"\n  Missing links:")
            for ml in linking_evaluation.missing_links:
                print(f"    - {ml}")

        if linking_evaluation.spurious_links:
            print(f"\n  Spurious links:")
            for sl in linking_evaluation.spurious_links:
                print(f"    - {sl}")

        if linking_evaluation.improvement_suggestions:
            print(f"\n  Suggestions:")
            for s in linking_evaluation.improvement_suggestions:
                print(f"    - {s}")

    except Exception as e:
        print(f"  [WARN] Linking judge failed: {e}")

else:
    print("[SKIP] No performance data to evaluate")

2026/02/09 11:05:43 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Evaluating linking quality with LLM judge...

LINKING EVALUATION RESULTS
  Material Identity Match:       4.5/5.0
  Performance Data Correctness:  5.0/5.0
  Completeness:                  4.0/5.0
  Format & Structure:            5.0/5.0
  Overall:                       4.6/5.0
  Confidence:                    high

  Failure flags: ['F8']

  Missing links:
    - Co9Mo3/MCM-41 in Figure 6

  Suggestions:
    - Investigate the cause of the missing link for 'Co9Mo3/MCM-41' in Figure 6. Check for potential errors in the synthesis extraction or series name recognition.
    - Improve the algorithm's ability to handle OCR errors or typos in series names, potentially by using fuzzy matching or contextual information to identify the most likely material.


: 

: 

: 

## Step 8: Build Final Results

In [ ]:
# Combine synthesis + performance + linking evaluation
final_results = []

for entry in all_syntheses:
    result = {
        "material": entry.material,
        "synthesis": entry.synthesis.model_dump() if entry.synthesis else None,
        "evaluation": entry.evaluation.model_dump() if entry.evaluation else None,
        "performance": (
            performance_data[entry.material].model_dump()
            if entry.material in performance_data
            else None
        ),
        "linking_evaluation": (
            linking_evaluation.model_dump() if linking_evaluation else None
        ),
    }
    final_results.append(result)

# Summary
materials_with_perf = [m for m in materials if m in performance_data]
materials_without_perf = [m for m in materials if m not in performance_data]

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"  Paper: {paper.name}")
print(f"  Total materials: {len(materials)}")
print(f"  Materials with synthesis: {sum(1 for r in final_results if r['synthesis'])}")
print(f"  Materials with performance: {len(materials_with_perf)}")
print(f"  Materials without performance: {len(materials_without_perf)}")
if plots:
    print(f"  Total plots extracted: {len(plots)}")
    print(f"  Plots linked: {len(plot_mappings)}")
if linking_evaluation:
    print(f"  Linking quality score: {linking_evaluation.scores.overall_score}/5.0")

FINAL SUMMARY
  Paper: 01_Duan_2012_MCM
  Total materials: 9
  Materials with synthesis: 9
  Materials with performance: 8
  Materials without performance: 1
  Total plots extracted: 8
  Plots linked: 2
  Linking quality score: 4.6/5.0


: 

: 

: 

In [ ]:
# Show one complete result
if final_results:
    # Find a result with both synthesis and performance
    example = next((r for r in final_results if r['synthesis'] and r['performance']), final_results[0])
    
    print("=" * 60)
    print(f"EXAMPLE RESULT: {example['material']}")
    print("=" * 60)
    print(json.dumps(example, indent=2, default=str)[:3000])
    if len(json.dumps(example, indent=2)) > 3000:
        print("... (truncated)")

EXAMPLE RESULT: Co/MCM-41
{
  "material": "Co/MCM-41",
  "synthesis": {
    "target_compound": "Co/MCM-41",
    "target_compound_type": "functional materials & catalysts",
    "synthesis_method": "wet impregnation",
    "starting_materials": [
      {
        "vendor": null,
        "name": "MCM-41",
        "amount": 126.0,
        "unit": "\u03bcm",
        "purity": null
      },
      {
        "vendor": null,
        "name": "Co(NO3)2\u00b76H2O",
        "amount": null,
        "unit": null,
        "purity": "ACS grade"
      },
      {
        "vendor": null,
        "name": "Deionized Water",
        "amount": null,
        "unit": null,
        "purity": null
      }
    ],
    "steps": [
      {
        "step_number": 1,
        "action": "impregnate",
        "description": "MCM-41 supported Co catalysts were prepared by incipient wetness impregnation with an aqueous solution of precursors (Co(NO3)2\u00b76H2O). The loading of the components is 5.0 wt.%.",
        "materials"

: 

: 

: 

## Step 9: Save Results

In [ ]:
import os
from llm_synthesis.utils.performance_utils import sanitize_filename

# Create output directory
paper_dir = os.path.join(OUTPUT_DIR, paper.id)
os.makedirs(paper_dir, exist_ok=True)

# Save individual material files (without linking_evaluation — that goes in summary)
for result in final_results:
    mat_result = {k: v for k, v in result.items() if k != "linking_evaluation"}
    mat_name = sanitize_filename(result["material"])
    mat_path = os.path.join(paper_dir, f"{mat_name}.json")
    with open(mat_path, "w") as f:
        json.dump(mat_result, f, indent=2, default=str)

# Save plot mappings
if plot_mappings:
    mappings_path = os.path.join(paper_dir, "performance_mappings.json")
    with open(mappings_path, "w") as f:
        json.dump([m.model_dump() for m in plot_mappings], f, indent=2)

# Base summary content (shared between LLM and human versions)
base_summary = {
    "paper_id": paper.id,
    "paper_name": paper.name,
    "total_materials": len(materials),
    "materials_with_performance": len(materials_with_perf),
    "materials_without_performance": len(materials_without_perf),
    "materials_list": materials,
    "materials_with_performance_list": materials_with_perf,
    "materials_without_performance_list": materials_without_perf,
    "total_plots_extracted": len(plots) if plots else 0,
    "plots_linked": len(plot_mappings),
}

# --- linking_summary_llm.json: summary + LLM evaluation ---
llm_summary = {**base_summary}
if linking_evaluation:
    llm_summary["linking_evaluation"] = linking_evaluation.model_dump()
else:
    llm_summary["linking_evaluation"] = None

llm_path = os.path.join(paper_dir, "linking_summary_llm.json")
with open(llm_path, "w") as f:
    json.dump(llm_summary, f, indent=2, default=str)

# --- linking_summary_human.json: summary + empty evaluation for annotation ---
human_summary = {**base_summary}
human_summary["linking_evaluation"] = {
    "reasoning": None,
    "scores": {
        "material_identity_score": None,
        "material_identity_reasoning": None,
        "performance_data_correctness_score": None,
        "performance_data_correctness_reasoning": None,
        "completeness_score": None,
        "completeness_reasoning": None,
        "format_structure_score": None,
        "format_structure_reasoning": None,
        "overall_score": None,
        "overall_reasoning": None,
    },
    "failure_flags": {
        "f1_name_mismatch": None,
        "f2_one_to_many_synthesis": None,
        "f3_many_to_one_figure": None,
        "f4_sample_code_failure": None,
        "f5_precursor_vs_product": None,
        "f6_characterization_confusion": None,
        "f7_dual_axis_error": None,
        "f8_false_negative": None,
        "f9_false_positive": None,
    },
    "confidence_level": None,
    "missing_links": None,
    "spurious_links": None,
    "improvement_suggestions": None,
}

human_path = os.path.join(paper_dir, "linking_summary_human.json")
with open(human_path, "w") as f:
    json.dump(human_summary, f, indent=2, default=str)

print(f"[OK] Results saved to: {paper_dir}/")
print(f"   - {len(final_results)} material files")
print(f"   - performance_mappings.json")
print(f"   - linking_summary_llm.json")
print(f"   - linking_summary_human.json")

[OK] Results saved to: results/01_Duan_2012_MCM/
   - 9 material files
   - performance_mappings.json
   - linking_summary_llm.json
   - linking_summary_human.json


: 

: 

: 

---

## Done!

You have successfully extracted:
- **Materials** from the paper
- **Synthesis procedures** for each material
- **Performance data** from plots, linked to specific materials

Check the output directory for the saved results.